In [2]:
%pip install duckdb

Note: you may need to restart the kernel to use updated packages.


### initial steps using duckdb to understand and validate the data

In [3]:
import duckdb
con = duckdb.connect()
RAW = "../data/raw/steam"

def src(t):
    return f"read_csv('{RAW}/{t}.csv', header=True, sample_size=-1)"

t, key = "players", "playerid"

con.sql(f"DESCRIBE SELECT * FROM {src(t)}").show() # for column names and primary key check
con.sql(f"SELECT COUNT(*) n, COUNT(DISTINCT {key}) n_key FROM {src(t)}").show() # same thing but for counting the rows this time
con.sql(f"SUMMARIZE SELECT * FROM {src(t)}").show() # checking null, min, max etc. for every column
con.sql(f"SELECT * FROM {src(t)} LIMIT 5").show() # for general looking

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ playerid    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ country     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ created     │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

┌────────┬────────┐
│   n    │ n_key  │
│ int64  │ int64  │
├────────┼────────┤
│ 424683 │ 424683 │
└────────┴────────┘

┌─────────────┬─────────────┬─────────────────────┬─────────────────────┬───────────────┬────────────────────────────┬────────────────────┬───────────────────────────┬────────────────────────────┬────────────────────────────┬────────┬─────────────────┐
│ column_name │ column_type │         min         │         max  

#### since we see 266 countries i suspect that (the result is the real number is 249 so it was the approx_unique guess. the data is clean)

In [4]:
con.sql(f"SELECT country, COUNT(*) n FROM {src(t)} GROUP BY 1 ORDER BY n").show(max_rows=30)

┌─────────────────────────────┬────────┐
│           country           │   n    │
│           varchar           │ int64  │
├─────────────────────────────┼────────┤
│ Sao Tome and Principe       │      8 │
│ Malawi                      │      9 │
│ Lesotho                     │      9 │
│ Montserrat                  │      9 │
│ Mauritania                  │     10 │
│ Benin                       │     10 │
│ Equatorial Guinea           │     10 │
│ Guinea                      │     12 │
│ Saint Martin (French part)  │     13 │
│ French Polynesia            │     13 │
│ French Guiana               │     13 │
│ Sint Maarten (Dutch part)   │     14 │
│ French Southern Territories │     14 │
│ Comoros                     │     15 │
│ Gambia                      │     15 │
│   ·                         │      · │
│   ·                         │      · │
│   ·                         │      · │
│ Portugal                    │   4662 │
│ Romania                     │   4756 │
│ Sweden        

In [5]:
t, key = "private_steamids", "playerid" 

con.sql(f"""
SELECT
  COUNT(*)                                   AS n_private,
  COUNT(p.playerid)                          AS in_players,
  COUNT(*) FILTER (p.country IS NULL)        AS country_null,
  COUNT(*) FILTER (p.created IS NULL)        AS created_null
FROM {src('private_steamids')} s
LEFT JOIN {src('players')} p USING (playerid)
""").show()

┌───────────┬────────────┬──────────────┬──────────────┐
│ n_private │ in_players │ country_null │ created_null │
│   int64   │   int64    │    int64     │    int64     │
├───────────┼────────────┼──────────────┼──────────────┤
│    227963 │     227963 │       112334 │        27480 │
└───────────┴────────────┴──────────────┴──────────────┘



In [6]:
con.sql(f"SELECT COUNT(*) n, COUNT(DISTINCT playerid) n_key FROM {src('private_steamids')}").show()

┌────────┬────────┐
│   n    │ n_key  │
│ int64  │ int64  │
├────────┼────────┤
│ 227963 │ 227963 │
└────────┴────────┘



In [7]:
t, key = "games", "gameid"

con.sql(f"DESCRIBE SELECT * FROM {src(t)}").show()
con.sql(f"SELECT COUNT(*) n, COUNT(DISTINCT {key}) n_key FROM {src(t)}").show()
con.sql(f"SUMMARIZE SELECT * FROM {src(t)}").show()
con.sql(f"SELECT * FROM {src(t)} LIMIT 5").show()

┌─────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name     │ column_type │  null   │   key   │ default │  extra  │
│       varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ gameid              │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ title               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ developers          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ publishers          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ genres              │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ supported_languages │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ release_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

┌───────┬───────┐
│   n   │ n_key │
│ int64 │ int64 │
├───────┼

In [8]:
con.sql(f"""
SELECT title ILIKE '%playtest%' AS is_playtest,
       genres IS NULL           AS no_genre,
       COUNT(*) n
FROM {src(t)} GROUP BY ALL ORDER BY ALL
""").show()

┌─────────────┬──────────┬───────┐
│ is_playtest │ no_genre │   n   │
│   boolean   │ boolean  │ int64 │
├─────────────┼──────────┼───────┤
│ false       │ false    │ 92691 │
│ false       │ true     │   260 │
│ true        │ false    │     7 │
│ true        │ true     │  5287 │
│ NULL        │ false    │     1 │
│ NULL        │ true     │     2 │
└─────────────┴──────────┴───────┘



In [9]:
con.sql(f"""
SELECT trim(g, ' ''"') AS genre, COUNT(*) n
FROM (SELECT unnest(string_split(trim(genres, '[]'), ', ')) g FROM {src(t)})
GROUP BY 1 ORDER BY n DESC
""").show(max_rows=40)

┌───────────────────────┬───────┐
│         genre         │   n   │
│        varchar        │ int64 │
├───────────────────────┼───────┤
│ Indie                 │ 65024 │
│ Casual                │ 39869 │
│ Action                │ 38301 │
│ Adventure             │ 36664 │
│ Simulation            │ 18899 │
│ Strategy              │ 18018 │
│ RPG                   │ 16851 │
│ Early Access          │  9496 │
│ Free To Play          │  8751 │
│ Sports                │  4092 │
│ Racing                │  3376 │
│ Massively Multiplayer │  2214 │
│ Utilities             │   880 │
│ Design & Illustration │   516 │
│ Violent               │   433 │
│ Animation & Modeling  │   410 │
│ Education             │   402 │
│ Video Production      │   305 │
│ Game Development      │   269 │
│ Gore                  │   266 │
│ Audio Production      │   206 │
│ Software Training     │   198 │
│ Photo Editing         │   153 │
│ Web Publishing        │   105 │
│ Nudity                │    99 │
│ Sexual Conte

In [10]:
t, key = "achievements", "achievementid"

con.sql(f"""
SELECT
  COUNT(*)                                                       AS n,
  COUNT(*) FILTER (split_part(achievementid, '_', 1) != CAST(gameid AS VARCHAR)) AS prefix_mismatch,
  COUNT(*) FILTER (g.gameid IS NULL)                             AS game_not_in_catalog
FROM {src('achievements')} a
LEFT JOIN (SELECT gameid FROM {src('games')}) g USING (gameid)
""").show()

┌─────────┬─────────────────┬─────────────────────┐
│    n    │ prefix_mismatch │ game_not_in_catalog │
│  int64  │      int64      │        int64        │
├─────────┼─────────────────┼─────────────────────┤
│ 1939027 │               0 │                3145 │
└─────────┴─────────────────┴─────────────────────┘



In [11]:
# let's see that which records does not have the 10 core game genres

con.sql(f"""
WITH g AS (
  SELECT gameid, list_transform(string_split(trim(genres, '[]'), ', '), x -> trim(x, ' ''"')) AS gl
  FROM {src('games')} WHERE genres IS NOT NULL
)
SELECT COUNT(*) AS non_game_only
FROM g
WHERE NOT list_has_any(gl, ['Indie','Casual','Action','Adventure','Simulation','Strategy',
                            'RPG','Sports','Racing','Massively Multiplayer'])
""").show()

┌───────────────┐
│ non_game_only │
│     int64     │
├───────────────┤
│          1495 │
└───────────────┘



In [12]:
con.sql(f"""
WITH g AS (
  SELECT gameid, list_transform(string_split(trim(genres, '[]'), ', '), x -> trim(x, ' ''"')) AS gl
  FROM {src('games')} WHERE genres IS NOT NULL
)
SELECT
  list_has_any(gl, ['Utilities','Design & Illustration','Animation & Modeling','Education',
                    'Video Production','Game Development','Audio Production','Software Training',
                    'Photo Editing','Web Publishing','Accounting',
                    'Movie','Documentary','360 Video','Tutorial','Short','Episodic']) AS has_software_media,
  list_has_any(gl, ['Early Access','Free To Play','Violent','Gore','Nudity','Sexual Content']) AS has_tag,
  COUNT(*) n
FROM g
WHERE NOT list_has_any(gl, ['Indie','Casual','Action','Adventure','Simulation','Strategy',
                            'RPG','Sports','Racing','Massively Multiplayer'])
GROUP BY ALL ORDER BY ALL
""").show()

┌────────────────────┬─────────┬───────┐
│ has_software_media │ has_tag │   n   │
│      boolean       │ boolean │ int64 │
├────────────────────┼─────────┼───────┤
│ false              │ true    │   162 │
│ true               │ false   │  1106 │
│ true               │ true    │   227 │
└────────────────────┴─────────┴───────┘



In [13]:
con.sql(f"""
WITH g AS (
  SELECT gameid, title, list_transform(string_split(trim(genres, '[]'), ', '), x -> trim(x, ' ''"')) AS gl
  FROM {src('games')} WHERE genres IS NOT NULL
)
SELECT title, gl FROM g
WHERE NOT list_has_any(gl, ['Indie','Casual','Action','Adventure','Simulation','Strategy',
                            'RPG','Sports','Racing','Massively Multiplayer'])
  AND list_has_any(gl, ['Utilities','Design & Illustration','Animation & Modeling','Education',
                        'Video Production','Game Development','Audio Production','Software Training',
                        'Photo Editing','Web Publishing','Accounting',
                        'Movie','Documentary','360 Video','Tutorial','Short','Episodic'])
  AND list_has_any(gl, ['Early Access','Free To Play','Violent','Gore','Nudity','Sexual Content'])
LIMIT 15
""").show(max_width=200)

┌─────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────┐
│            title            │                                            gl                                            │
│           varchar           │                                        varchar[]                                         │
├─────────────────────────────┼──────────────────────────────────────────────────────────────────────────────────────────┤
│ Driver4VR                   │ [Utilities, Early Access]                                                                │
│ Defold                      │ [Free To Play, Game Development]                                                         │
│ SYNTHSPACE                  │ [Audio Production, Education, Software Training, Early Access]                           │
│ CatTuber                    │ [Animation & Modeling, Utilities, Video Production, Early Access]                        │
│ In-Home Stream

In [14]:
CORE = ['Indie','Casual','Action','Adventure','Simulation','Strategy',
        'RPG','Sports','Racing','Massively Multiplayer']
SOFT = ['Utilities','Design & Illustration','Animation & Modeling','Education',
        'Video Production','Game Development','Audio Production','Software Training',
        'Photo Editing','Web Publishing','Accounting',
        'Movie','Documentary','360 Video','Tutorial','Short','Episodic']

con.sql(f"""
CREATE OR REPLACE VIEW games_clean AS
WITH g AS (
  SELECT *, list_transform(string_split(trim(genres, '[]'), ', '), x -> trim(x, ' ''"')) AS gl
  FROM {src('games')}
)
SELECT * EXCLUDE (gl) FROM g
WHERE title IS NOT NULL
  AND title NOT ILIKE '%playtest%'
  AND NOT (genres IS NOT NULL
           AND NOT list_has_any(gl, {CORE})
           AND list_has_any(gl, {SOFT}))
""")

con.sql("SELECT COUNT(*) FROM games_clean").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        91618 │
└──────────────┘



In [15]:
con.sql("COPY games_clean TO '../data/processed/games_clean.parquet' (FORMAT parquet)")

In [16]:
t, key = "achievements", "achievementid"

con.sql(f"DESCRIBE SELECT * FROM {src(t)}").show()
con.sql(f"SELECT COUNT(*) n, COUNT(DISTINCT {key}) n_key FROM {src(t)}").show()
con.sql(f"SUMMARIZE SELECT * FROM {src(t)}").show()
con.sql(f"SELECT * FROM {src(t)} LIMIT 5").show()

con.sql(f"""
SELECT COUNT(DISTINCT gameid) AS n_missing_games
FROM {src('achievements')} a
ANTI JOIN {src('games')} g USING (gameid)
""").show()

┌───────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name  │ column_type │  null   │   key   │ default │  extra  │
│    varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ achievementid │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ gameid        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ title         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ description   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└───────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

┌─────────┬─────────┐
│    n    │  n_key  │
│  int64  │  int64  │
├─────────┼─────────┤
│ 1939027 │ 1939027 │
└─────────┴─────────┘

┌───────────────┬─────────────┬───────────────────────────────────┬────────────────────────┬───────────────┬────────────────────┬───────────────────┬─────────┬─────────┬─────────┬─────────┬─────────────────┐
│  column_

In [17]:
con.sql(f"""
WITH per_game AS (
  SELECT gameid, COUNT(*) n_ach FROM {src('achievements')} GROUP BY gameid
)
SELECT COUNT(*) AS n_games,
       MIN(n_ach), quantile_cont(n_ach, 0.5) AS median, quantile_cont(n_ach, 0.99) AS p99, MAX(n_ach)
FROM per_game
""").show()

┌─────────┬────────────┬────────┬────────┬────────────┐
│ n_games │ min(n_ach) │ median │  p99   │ max(n_ach) │
│  int64  │   int64    │ double │ double │   int64    │
├─────────┼────────────┼────────┼────────┼────────────┤
│   50548 │          1 │   18.0 │  152.0 │      10979 │
└─────────┴────────────┴────────┴────────┴────────────┘



In [18]:
con.sql(f"""
SELECT a.gameid, g.title, COUNT(*) n_ach
FROM {src('achievements')} a
LEFT JOIN {src('games')} g USING (gameid)
GROUP BY ALL ORDER BY n_ach DESC LIMIT 15
""").show(max_width=200)

┌────────┬───────────────────────────────────────────────────────────────────────┬───────┐
│ gameid │                                 title                                 │ n_ach │
│ int64  │                                varchar                                │ int64 │
├────────┼───────────────────────────────────────────────────────────────────────┼───────┤
│ 664290 │ The Mexican Dream                                                     │ 10979 │
│ 573060 │ LOGistICAL                                                            │  9821 │
│ 640310 │ Drive-By Hero                                                         │  5394 │
│ 699590 │ Trivia Vault: Science & History Trivia 2                              │  5000 │
│ 821390 │ Achievement Lurker: Another one bites the dust                        │  5000 │
│ 877360 │ 18+                                                                   │  5000 │
│ 818480 │ Eternal Man: Mountain                                                 │  5000 │

In [19]:
t, key = "prices", "(gameid, date_acquired)"

con.sql(f"DESCRIBE SELECT * FROM {src(t)}").show()
con.sql(f"SELECT COUNT(*) n, COUNT(DISTINCT {key}) n_key FROM {src(t)}").show()
con.sql(f"SUMMARIZE SELECT * FROM {src(t)}").show()
con.sql(f"SELECT * FROM {src(t)} LIMIT 5").show()

con.sql(f"""
SELECT MIN(date_acquired), MAX(date_acquired), COUNT(DISTINCT date_acquired) n_dates,
       COUNT(DISTINCT gameid) n_games
FROM {src('prices')}
""").show()

┌───────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name  │ column_type │  null   │   key   │ default │  extra  │
│    varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ gameid        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ usd           │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ eur           │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ gbp           │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ jpy           │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ rub           │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ date_acquired │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
└───────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

┌─────────┬─────────┐
│    n    │  n_key  │
│  int64  │  int64  │
├─────────┼─────────┤
│ 4414273 │ 4414273 │
└─────────┴─────────┘

┌─

In [20]:
con.sql(f"""
SELECT COUNT(DISTINCT gameid) FROM {src('prices')} p
ANTI JOIN {src('games')} g USING (gameid)
""").show()

┌────────────────────────┐
│ count(DISTINCT gameid) │
│         int64          │
├────────────────────────┤
│                    217 │
└────────────────────────┘



In [ ]:
t, key = "purchased_games", "playerid"

con.sql(f"DESCRIBE SELECT * FROM {src(t)}").show()
con.sql(f"SELECT COUNT(*) n, COUNT(DISTINCT {key}) n_key FROM {src(t)}").show()
con.sql(f"SUMMARIZE SELECT * FROM {src(t)}").show()
con.sql(f"SELECT * FROM {src(t)} LIMIT 5").show()

con.sql(f"""
WITH pg AS (
  SELECT playerid, len(CAST(library AS BIGINT[])) AS n_games
  FROM {src('purchased_games')}
)
SELECT
  (s.playerid IS NOT NULL)          AS is_private,
  COUNT(*)                          AS n_players,
  COUNT(*) FILTER (n_games = 0)     AS empty_library,
  quantile_cont(n_games, 0.5)       AS median_games,
  MAX(n_games)                      AS max_games
FROM pg
LEFT JOIN {src('private_steamids')} s USING (playerid)
GROUP BY ALL
""").show()

In [22]:
con.sql(f"""
SELECT
  (s.playerid IS NOT NULL)                     AS is_private,
  COUNT(*)                                     AS n_players,
  COUNT(*) FILTER (p.library IS NULL)          AS null_library,
  COUNT(*) FILTER (p.library = '[]')           AS empty_library,
  COUNT(*) FILTER (p.library IS NOT NULL AND p.library != '[]') AS has_library
FROM {src('purchased_games')} p
LEFT JOIN {src('private_steamids')} s USING (playerid)
GROUP BY ALL
""").show()


┌────────────┬───────────┬──────────────┬───────────────┬─────────────┐
│ is_private │ n_players │ null_library │ empty_library │ has_library │
│  boolean   │   int64   │    int64     │     int64     │    int64    │
├────────────┼───────────┼──────────────┼───────────────┼─────────────┤
│ false      │     49836 │        19705 │             0 │       30131 │
│ true       │     52712 │        35902 │             0 │       16810 │
└────────────┴───────────┴──────────────┴───────────────┴─────────────┘



# so far:

- bu noktaya kadar tam olarak 46941 adet dolu kütüphaneli oyuncu ortaya çıktı.
- boş kütüphane yok ("[]" şeklinde) ama null mevcut, bu da demek oluyor ki oyunu yok değil verisi yok.
- gizli olanların 16810'unun kütüphanesi mevcut.

In [23]:
con.sql(f"""
WITH lib AS (
  SELECT playerid FROM {src('purchased_games')} WHERE library IS NOT NULL
),
hist AS (
  SELECT DISTINCT playerid FROM {src('history')}
)
SELECT
  (SELECT COUNT(*) FROM lib)                                   AS n_lib,
  (SELECT COUNT(*) FROM hist)                                  AS n_hist,
  (SELECT COUNT(*) FROM lib JOIN hist USING (playerid))        AS both_,
  (SELECT COUNT(*) FROM lib ANTI JOIN hist USING (playerid))   AS lib_only,
  (SELECT COUNT(*) FROM hist ANTI JOIN lib USING (playerid))   AS hist_only
""").show()


┌───────┬────────┬───────┬──────────┬───────────┐
│ n_lib │ n_hist │ both_ │ lib_only │ hist_only │
│ int64 │ int64  │ int64 │  int64   │   int64   │
├───────┼────────┼───────┼──────────┼───────────┤
│ 46941 │   4838 │  4838 │    42103 │         0 │
└───────┴────────┴───────┴──────────┴───────────┘



şimdi history'de başarım sahibi olan 4838 oyuncu var. başarım tamamlama oranı bu 4838 oyuncu için hesaplanabiliyor. bu da elimizdeki 46941 oyuncunun yaklaşık olarak %10'una denk geliyor. 424 bin oyuncunun da %1'i. bundan sonra reviews tablosundaki oyuncuların lib ve history ile örtüşmesine bakacağız. ayrıca history'deki oyuncu başına row count'a ve date range'e de bakılmalı

In [24]:
con.sql(f"""
WITH lib AS (SELECT playerid FROM {src('purchased_games')} WHERE library IS NOT NULL),
rev AS (SELECT playerid, COUNT(*) n_rev, MIN(posted) first_post, MAX(posted) last_post
        FROM {src('reviews')} GROUP BY playerid)
SELECT
  (SELECT COUNT(*) FROM rev)                              AS n_reviewers,
  (SELECT COUNT(*) FROM rev JOIN lib USING (playerid))    AS rev_and_lib,
  (SELECT MIN(first_post) FROM rev)                       AS min_posted,
  (SELECT MAX(last_post)  FROM rev)                       AS max_posted
""").show()

con.sql(f"""
SELECT COUNT(*) n_rows, COUNT(DISTINCT playerid) n_players,
       MIN(date_acquired) min_d, MAX(date_acquired) max_d,
       round(COUNT(*) * 1.0 / COUNT(DISTINCT playerid), 1) AS rows_per_player
FROM {src('history')}
""").show()


┌─────────────┬─────────────┬────────────┬────────────┐
│ n_reviewers │ rev_and_lib │ min_posted │ max_posted │
│    int64    │    int64    │    date    │    date    │
├─────────────┼─────────────┼────────────┼────────────┤
│      196698 │       30130 │ 2010-10-15 │ 2025-01-09 │
└─────────────┴─────────────┴────────────┴────────────┘

┌──────────┬───────────┬─────────────────────┬─────────────────────┬─────────────────┐
│  n_rows  │ n_players │        min_d        │        max_d        │ rows_per_player │
│  int64   │   int64   │      timestamp      │      timestamp      │     double      │
├──────────┼───────────┼─────────────────────┼─────────────────────┼─────────────────┤
│ 10693879 │      4838 │ 2008-09-13 01:37:54 │ 2024-12-15 19:42:05 │          2210.4 │
└──────────┴───────────┴─────────────────────┴─────────────────────┴─────────────────┘



In [25]:
con.sql(f"""
WITH lib AS (SELECT playerid FROM {src('purchased_games')} WHERE library IS NOT NULL),
rev AS (SELECT playerid, COUNT(*) n_rev FROM {src('reviews')} GROUP BY playerid)
SELECT
  (SELECT COUNT(*) FROM (SELECT playerid FROM lib UNION SELECT playerid FROM rev)) AS union_players,
  (SELECT median(n_rev) FROM rev)                                AS med_rev,
  (SELECT quantile_cont(n_rev, 0.99) FROM rev)                   AS p99_rev,
  (SELECT COUNT(*) FROM rev WHERE n_rev = 1)                     AS single_rev_players,
  (SELECT COUNT(*) FROM rev WHERE n_rev >= 3)                    AS ge3_rev_players
""").show()

con.sql(f"""
WITH r AS (SELECT playerid, gameid FROM {src('reviews')}),
l AS (
  SELECT playerid, unnest(CAST(library AS BIGINT[])) AS gameid
  FROM {src('purchased_games')} WHERE library IS NOT NULL
)
SELECT COUNT(*) n_reviews_with_lib_player,
       COUNT(*) FILTER (l.gameid IS NOT NULL) AS review_game_in_library
FROM r
JOIN (SELECT DISTINCT playerid FROM l) USING (playerid)
LEFT JOIN l USING (playerid, gameid)
""").show()


┌───────────────┬─────────┬─────────┬────────────────────┬─────────────────┐
│ union_players │ med_rev │ p99_rev │ single_rev_players │ ge3_rev_players │
│     int64     │ double  │ double  │       int64        │      int64      │
├───────────────┼─────────┼─────────┼────────────────────┼─────────────────┤
│        213509 │     2.0 │    60.0 │              79193 │           84451 │
└───────────────┴─────────┴─────────┴────────────────────┴─────────────────┘

┌───────────────────────────┬────────────────────────┐
│ n_reviews_with_lib_player │ review_game_in_library │
│           int64           │         int64          │
├───────────────────────────┼────────────────────────┤
│                    256859 │                 228918 │
└───────────────────────────┴────────────────────────┘



etkileşim sahibi oyuncu sayımız şu an için 213 bin oldu. iki kaynağı birleştireceğiz. yorum yapanların neredeyse yarısı tek yoruma sahip. yorumların %11'i kütüphanede ait olunmayan bir item'a ait, muhtemelen iade edilmiş veya steam marketplace'ten kaldırılmış yine yorumlayabiliriz şu an için.

bunun dışında leakage riskimiz mevcut, time-based split'te bir oyuncunun yorumunu test verisine ayırırsak o oyun %89 ihtimalle kütüphanede de duruyor. "library owner" signal olarak train'e girerse test verisi train'e girmiş olur ve leakage riskimiz artar. o yüzden çözüm olarak da split sonrası test öğelerini library özelliklerinden de çıkaracağız.

In [26]:
con.sql(f"""
WITH per_game AS (
  SELECT gameid, COUNT(*) n_ach FROM {src('achievements')} GROUP BY gameid
)
SELECT
  COUNT(*) FILTER (n_ach > 152)                    AS gt_152,
  COUNT(*) FILTER (n_ach BETWEEN 153 AND 500)      AS b_153_500,
  COUNT(*) FILTER (n_ach BETWEEN 501 AND 1000)     AS b_501_1000,
  COUNT(*) FILTER (n_ach > 1000)                   AS gt_1000,
  COUNT(*) FILTER (n_ach = 5000)                   AS eq_5000
FROM per_game
""").show()


┌────────┬───────────┬────────────┬─────────┬─────────┐
│ gt_152 │ b_153_500 │ b_501_1000 │ gt_1000 │ eq_5000 │
│ int64  │   int64   │   int64    │  int64  │  int64  │
├────────┼───────────┼────────────┼─────────┼─────────┤
│    505 │       290 │         55 │     160 │      77 │
└────────┴───────────┴────────────┴─────────┴─────────┘



In [27]:
con.sql(f"""
SELECT a.gameid, g.title, COUNT(*) n_ach
FROM {src('achievements')} a
LEFT JOIN {src('games')} g USING (gameid)
GROUP BY ALL
HAVING COUNT(*) BETWEEN 501 AND 1000
ORDER BY n_ach DESC LIMIT 20
""").show(max_width=200)


┌─────────┬─────────────────────────────────────────────┬───────┐
│ gameid  │                    title                    │ n_ach │
│  int64  │                   varchar                   │ int64 │
├─────────┼─────────────────────────────────────────────┼───────┤
│  811250 │ Content Creator Simulator                   │  1000 │
│  710780 │ Sky Jump                                    │   998 │
│  828960 │ Blonde Driver                               │   994 │
│  863980 │ ORIX!                                       │   952 │
│  630060 │ 寇莎梅特：困世迷情 Consummate:Missing World │   888 │
│  608800 │ Guns of Icarus Alliance                     │   849 │
│  209080 │ Guns of Icarus Online                       │   849 │
│  772470 │ 8-in-1 IQ Scale Bundle                      │   830 │
│  668560 │ Thomaz                                      │   810 │
│ 1523370 │ A.D. 2047                                   │   804 │
│  617670 │ Zup! S                                      │   800 │
│  816190 │ Draw It

In [28]:
con.sql(f"""
WITH n AS (
  SELECT playerid, len(CAST(library AS BIGINT[])) AS n_games
  FROM {src('purchased_games')} WHERE library IS NOT NULL
)
SELECT min(n_games) mn, median(n_games) med,
       quantile_cont(n_games, 0.90) p90, quantile_cont(n_games, 0.99) p99,
       quantile_cont(n_games, 0.999) p999, max(n_games) mx,
       COUNT(*) FILTER (n_games = 0) AS zero_games,
       COUNT(*) FILTER (n_games > 1000) AS gt1000
FROM n
""").show()

┌───────┬────────┬────────┬────────────────────┬────────────────────┬───────┬────────────┬────────┐
│  mn   │  med   │  p90   │        p99         │        p999        │  mx   │ zero_games │ gt1000 │
│ int64 │ double │ double │       double       │       double       │ int64 │   int64    │ int64  │
├───────┼────────┼────────┼────────────────────┼────────────────────┼───────┼────────────┼────────┤
│     1 │   76.0 │  427.0 │ 3137.5999999999985 │ 12152.659999999974 │ 32463 │          0 │   1703 │
└───────┴────────┴────────┴────────────────────┴────────────────────┴───────┴────────────┴────────┘



In [29]:
con.sql(f"""
WITH n AS (
  SELECT playerid, len(CAST(library AS BIGINT[])) AS n_games
  FROM {src('purchased_games')} WHERE library IS NOT NULL
),
rev AS (SELECT DISTINCT playerid FROM {src('reviews')})
SELECT
  sum(n_games) FILTER (n_games > 1000) * 1.0 / sum(n_games)  AS share_of_rows_gt1000,
  sum(n_games) FILTER (n_games > 3000) * 1.0 / sum(n_games)  AS share_of_rows_gt3000,
  COUNT(*) FILTER (n_games > 1000 AND r.playerid IS NOT NULL) AS gt1000_reviewers,
  COUNT(*) FILTER (n_games BETWEEN 501 AND 1000)              AS b501_1000
FROM n LEFT JOIN rev r USING (playerid)
""").show()


┌──────────────────────┬──────────────────────┬──────────────────┬───────────┐
│ share_of_rows_gt1000 │ share_of_rows_gt3000 │ gt1000_reviewers │ b501_1000 │
│        double        │        double        │      int64       │   int64   │
├──────────────────────┼──────────────────────┼──────────────────┼───────────┤
│   0.4702969210962352 │   0.2957382791908313 │             1543 │      2172 │
└──────────────────────┴──────────────────────┴──────────────────┴───────────┘



In [30]:
t, key = "reviews", "reviewid"

con.sql(f"DESCRIBE SELECT * FROM {src(t)}").show()
con.sql(f"SELECT COUNT(*) n, COUNT(DISTINCT {key}) n_key FROM {src(t)}").show()
con.sql(f"SELECT reviewid, playerid, gameid, left(review, 60) AS review_head, helpful, funny, awards, posted FROM {src(t)} LIMIT 5").show(max_width=200)

con.sql(f"""
SELECT COUNT(*) AS n_rows,
       COUNT(*) FILTER (review IS NULL OR trim(review) = '') AS empty_review,
       COUNT(*) FILTER (posted IS NULL)                       AS null_posted,
       COUNT(DISTINCT gameid)                                 AS n_games,
       COUNT(DISTINCT (playerid, gameid))                     AS n_pairs
FROM {src(t)}
""").show()

con.sql(f"""
SELECT COUNT(DISTINCT gameid) AS reviewed_games_not_in_catalog
FROM {src(t)} r ANTI JOIN {src('games')} g USING (gameid)
""").show()


┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ reviewid    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ playerid    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gameid      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ review      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ helpful     │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ funny       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ awards      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ posted      │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

┌─────────┬─────────┐
│    n    │  n_key  │
│  int64  │  int64  │
├─────────┼─────────┤
│

boş metinli 2,096 yorum yine bir etkileşim sayılır, çünkü oyuncu o oyuna yorum bırakmış diyebiliriz. 161 katalog dışı oyunu etkileşimlerden düşeceğiz, kaç yorum satırının etkilendiğini henüz bilmiyoruz. bunun dışında tabloda yorumun olumlu veya olumsuz olduğunu gösteren bir column yok. henüz bakmadıklarımızda yıllara göre bir dağılım var. games_clean ile de örtüşmeye bakacağız, yani katalogtan çıkardığımız oyunlar etkileşimlerden ne kadar eksiltiyor? özellikle de hem yorumlar hem de kütüphane için.

In [31]:
con.sql(f"""
SELECT year(posted) AS yr, COUNT(*) n_reviews, COUNT(DISTINCT playerid) n_players
FROM {src('reviews')}
GROUP BY 1 ORDER BY 1
""").show(max_rows=30)

con.sql(f"""
SELECT
  COUNT(*)                                                     AS n_reviews,
  COUNT(*) FILTER (c.gameid IS NOT NULL)                       AS in_games_clean,
  COUNT(*) FILTER (c.gameid IS NULL)                           AS dropped,
  round(COUNT(*) FILTER (c.gameid IS NULL) * 100.0 / COUNT(*), 2) AS dropped_pct
FROM {src('reviews')} r
LEFT JOIN read_parquet('../data/processed/games_clean.parquet') c USING (gameid)
""").show()


┌───────┬───────────┬───────────┐
│  yr   │ n_reviews │ n_players │
│ int64 │   int64   │   int64   │
├───────┼───────────┼───────────┤
│  2010 │       733 │       408 │
│  2011 │      3203 │      1552 │
│  2012 │      3883 │      2070 │
│  2013 │     14050 │      6439 │
│  2014 │     40207 │     19429 │
│  2015 │     51170 │     24154 │
│  2016 │     69458 │     33131 │
│  2017 │     83406 │     37641 │
│  2018 │     80324 │     34410 │
│  2019 │    108268 │     49023 │
│  2020 │    147404 │     61664 │
│  2021 │    140577 │     55173 │
│  2022 │    133043 │     50325 │
│  2023 │    144169 │     52267 │
│  2024 │    183087 │     53148 │
│  2025 │      1552 │      1079 │
└───────┴───────────┴───────────┘
  16 rows             3 columns

┌───────────┬────────────────┬─────────┬─────────────┐
│ n_reviews │ in_games_clean │ dropped │ dropped_pct │
│   int64   │     int64      │  int64  │   double    │
├───────────┼────────────────┼─────────┼─────────────┤
│   1204534 │        1186291 │   

training için 2024-01-01 öncesi, test içinse 2024-01-01 ve sonrasını alabiliriz. böylelikle de yorumların %85'i eğitimde kalır, test yeterince büyük, ve tek bir kesim tarihi olduğu için gerçek zamanı taklit etmiş olur

In [32]:
con.sql(f"""
WITH lib AS (SELECT playerid FROM {src('purchased_games')} WHERE library IS NOT NULL),
r AS (
  SELECT r.playerid, r.gameid, r.posted
  FROM {src('reviews')} r
  SEMI JOIN read_parquet('../data/processed/games_clean.parquet') c USING (gameid)
),
test AS (
  SELECT playerid, COUNT(*) n_test FROM r WHERE posted >= DATE '2024-01-01' GROUP BY playerid
),
train AS (
  SELECT playerid, COUNT(*) n_train FROM r WHERE posted < DATE '2024-01-01' GROUP BY playerid
)
SELECT
  COUNT(*)                                                                 AS test_players,
  COUNT(*) FILTER (tr.playerid IS NOT NULL)                                AS has_train_review,
  COUNT(*) FILTER (l.playerid  IS NOT NULL)                                AS has_library,
  COUNT(*) FILTER (tr.playerid IS NOT NULL OR l.playerid IS NOT NULL)      AS warm,
  COUNT(*) FILTER ((tr.playerid IS NOT NULL OR l.playerid IS NOT NULL) AND tr.n_train >= 2) AS warm_ge2_train
FROM test t
LEFT JOIN train tr USING (playerid)
LEFT JOIN lib l USING (playerid)
""").show()


┌──────────────┬──────────────────┬─────────────┬───────┬────────────────┐
│ test_players │ has_train_review │ has_library │ warm  │ warm_ge2_train │
│    int64     │      int64       │    int64    │ int64 │     int64      │
├──────────────┼──────────────────┼─────────────┼───────┼────────────────┤
│        52978 │            43417 │        9282 │ 44705 │          36721 │
└──────────────┴──────────────────┴─────────────┴───────┴────────────────┘



interaction table kurulumu:

In [33]:
con.sql(f"""
COPY (
  WITH lib AS (
    SELECT playerid, unnest(CAST(library AS BIGINT[])) AS gameid
    FROM {src('purchased_games')}
    WHERE library IS NOT NULL
  ),
  rev AS (
    SELECT playerid, gameid, posted FROM {src('reviews')}
  ),
  gc AS (SELECT gameid FROM read_parquet('../data/processed/games_clean.parquet'))
  SELECT
    COALESCE(l.playerid, r.playerid) AS playerid,
    COALESCE(l.gameid,   r.gameid)   AS gameid,
    (l.gameid IS NOT NULL)           AS in_library,
    (r.gameid IS NOT NULL)           AS reviewed,
    r.posted                         AS posted
  FROM (SELECT DISTINCT playerid, gameid FROM lib) l
  FULL OUTER JOIN rev r USING (playerid, gameid)
  SEMI JOIN gc ON gc.gameid = COALESCE(l.gameid, r.gameid)
) TO '../data/interim/interactions.parquet' (FORMAT parquet)
""")


In [34]:
I = "read_parquet('../data/interim/interactions.parquet')"

con.sql(f"""
SELECT COUNT(*)                                   AS n_rows,
       COUNT(DISTINCT playerid)                   AS n_players,
       COUNT(DISTINCT gameid)                     AS n_games,
       COUNT(*) FILTER (in_library AND NOT reviewed) AS lib_only,
       COUNT(*) FILTER (reviewed AND NOT in_library) AS rev_only,
       COUNT(*) FILTER (in_library AND reviewed)     AS both_
FROM {I}
""").show()

con.sql(f"""
WITH g AS (SELECT gameid, COUNT(*) n FROM {I} GROUP BY gameid)
SELECT COUNT(*) n_games,
       COUNT(*) FILTER (n < 5)   AS lt5,
       COUNT(*) FILTER (n < 10)  AS lt10,
       COUNT(*) FILTER (n < 20)  AS lt20,
       median(n) AS med, quantile_cont(n, 0.9) AS p90
FROM g
""").show()


┌──────────┬───────────┬─────────┬──────────┬──────────┬────────┐
│  n_rows  │ n_players │ n_games │ lib_only │ rev_only │ both_  │
│  int64   │   int64   │  int64  │  int64   │  int64   │ int64  │
├──────────┼───────────┼─────────┼──────────┼──────────┼────────┤
│ 10996537 │    212864 │   58696 │  9810246 │   960644 │ 225647 │
└──────────┴───────────┴─────────┴──────────┴──────────┴────────┘

┌─────────┬───────┬───────┬───────┬────────┬────────┐
│ n_games │  lt5  │ lt10  │ lt20  │  med   │  p90   │
│  int64  │ int64 │ int64 │ int64 │ double │ double │
├─────────┼───────┼───────┼───────┼────────┼────────┤
│   58696 │ 22581 │ 27292 │ 31898 │   13.0 │  381.0 │
└─────────┴───────┴───────┴───────┴────────┴────────┘

